In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
path = kagglehub.competition_download('porto-seguro-safe-driver-prediction')

print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/porto-seguro-safe-driver-prediction


In [ ]:
import os
os.listdir(path)
train_path=os.path.join(path,'train.csv')
test_path=os.path.join(path,'test.csv')
submissions_path=os.path.join(path , 'submissions.csv')

In [ ]:
#@title Reading all the datasets:

import pandas as pd
import numpy as np
df_train=pd.read_csv(train_path)
df_test=pd.read_csv(test_path)


In [ ]:
pd.set_option('display.max_columns', None)
df_train.head(5)

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.718070,10,1,-1,0,1,4,1,0,0,1,12,2,0.400000,0.883679,0.370810,3.605551,0.6,0.5,0.2,3,1,10,1,10,1,5,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,0,0,0,0,0,0,3,0,0,1,0.8,0.4,0.766078,11,1,-1,0,-1,11,1,1,2,1,19,3,0.316228,0.618817,0.388716,2.449490,0.3,0.1,0.3,2,1,9,5,8,1,7,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,-1.000000,7,1,-1,0,-1,14,1,1,2,1,60,1,0.316228,0.641586,0.347275,3.316625,0.5,0.7,0.1,2,2,9,1,8,2,7,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,0,0,0,0,0,0,8,1,0,0,0.9,0.2,0.580948,7,1,0,0,1,11,1,1,3,1,104,1,0.374166,0.542949,0.294958,2.000000,0.6,0.9,0.1,2,4,7,1,8,4,2,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.840759,11,1,-1,0,-1,14,1,1,2,1,82,3,0.316070,0.565832,0.365103,2.000000,0.4,0.6,0.0,2,2,6,3,10,2,12,3,1,1,3,0,0,0,1,1,0


A neat thing with this dataset is that the column names include the suffix 'bin' if its a binary feature and 'cat' if it is categorical

In [ ]:
#@title Looking for missing values:

import numpy as np

print("No of total missing values: ", df_train.isnull().sum().sum())
print("Shape of dataframe: ", df_train.shape)

No of total missing values:  0
Shape of dataframe:  (595212, 59)


We see a massive dataset of around 595,000 rows and almost 60 features , and all the feature names are not clear indicators. We only know if they are binary or categorical.

Also , the missing values are encoded as '-1'  , thus we must carefully check these as well

In [ ]:
missing_counts=(df_train==-1).sum()
missing_counts

,0
id,0
target,0
ps_ind_01,0
ps_ind_02_cat,216
ps_ind_03,0
ps_ind_04_cat,83
ps_ind_05_cat,5809
ps_ind_06_bin,0
ps_ind_07_bin,0
ps_ind_08_bin,0


We can see that we can instantly drop ps_car_03_cat(~ 65% missing) and ps_car_05_cat(~45% missing) as imputing these will just lead to too much noise.
The remaining can be imputed.

In [ ]:
#@title Checking for random missingness(MCAR):

for col in ['ps_car_14', 'ps_reg_03', 'ps_car_07_cat']:
  mask=df_train[col]==-1
  print(col , df_train.loc[mask, 'target'].mean(), df_train.loc[~mask, 'target'].mean())
#dropping the previously mentioned columns with extremely high amounts of missing vals
df_train.drop(columns=['ps_car_03_cat', 'ps_car_05_cat'])


ps_car_14 0.04042702956358517 0.03614058835451834
ps_reg_03 0.028393274691014363 0.03822829476448383
ps_car_07_cat 0.07816171990599705 0.035626487220822206


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_04_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.718070,10,1,0,4,1,0,0,1,12,2,0.400000,0.883679,0.370810,3.605551,0.6,0.5,0.2,3,1,10,1,10,1,5,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,0,0,0,0,0,0,3,0,0,1,0.8,0.4,0.766078,11,1,0,11,1,1,2,1,19,3,0.316228,0.618817,0.388716,2.449490,0.3,0.1,0.3,2,1,9,5,8,1,7,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,-1.000000,7,1,0,14,1,1,2,1,60,1,0.316228,0.641586,0.347275,3.316625,0.5,0.7,0.1,2,2,9,1,8,2,7,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,0,0,0,0,0,0,8,1,0,0,0.9,0.2,0.580948,7,1,0,11,1,1,3,1,104,1,0.374166,0.542949,0.294958,2.000000,0.6,0.9,0.1,2,4,7,1,8,4,2,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.840759,11,1,0,14,1,1,2,1,82,3,0.316070,0.565832,0.365103,2.000000,0.4,0.6,0.0,2,2,6,3,10,2,12,3,1,1,3,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595207,1488013,0,3,1,10,0,0,0,0,0,1,0,0,0,0,0,13,1,0,0,0.5,0.3,0.692820,10,1,0,1,1,1,0,1,31,3,0.374166,0.684631,0.385487,2.645751,0.4,0.5,0.3,3,0,9,0,9,1,12,4,1,9,6,0,1,1,0,1,1
595208,1488016,0,5,1,3,0,0,0,0,0,1,0,0,0,0,0,6,1,0,0,0.9,0.7,1.382027,9,1,0,15,0,0,2,1,63,2,0.387298,0.972145,-1.000000,3.605551,0.2,0.2,0.0,2,4,8,6,8,2,12,4,1,3,8,1,0,1,0,1,1
595209,1488017,0,1,1,10,0,0,1,0,0,0,0,0,0,0,0,12,1,0,0,0.9,0.2,0.659071,7,1,0,1,1,1,2,1,31,3,0.397492,0.596373,0.398748,1.732051,0.4,0.0,0.3,3,2,7,4,8,0,10,3,2,2,6,0,0,1,0,0,0
595210,1488021,0,5,2,3,1,0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.9,0.4,0.698212,11,1,0,11,1,1,2,1,101,3,0.374166,0.764434,0.384968,3.162278,0.0,0.7,0.0,4,0,9,4,9,2,11,4,1,4,2,0,1,1,1,0,0


In [ ]:
small_missing_cols = ['ps_car_01_cat', 'ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_09_cat']
any_missing = (df_train[small_missing_cols] == -1).any(axis=1)
df_train = df_train[~any_missing]

Here we are trying to evaluate is the fact that a value is missing related to the target variable.



So here , we are taking the mean of the target values where 'col' is missing and not missing , and comparing these. If these are close ,it means that the value missing isnt related to the target value.


Here we see that for the columns considered(as they have a high number of missing values), there is some level of correlation b/w the target and missing values. Thus it is better to impute and then flag the missing values. If there were no correlation(the values are almost / are the same) , we would simply impute in place.

In [ ]:
#@title Imputing and flagging out the missing values:
for col in ['ps_car_14', 'ps_reg_03', 'ps_car_07_cat']:
  df_train[f'{col}_missing']= (df_train[col]==-1).astype(int)
  med_val=df_train.loc[df_train[col]!=-1, col].median()
  df_train[col]=df_train[col].replace(-1 , med_val)

In [ ]:
#@title Checking class imbalance in target:

print(df_train['target'].value_counts(normalize=True))

target
0    0.964039
1    0.035961
Name: proportion, dtype: float64


Here we see that the dataset is extremely unbalanced with only ~3.4% +ve observations and ~96.6% -ve observations. Thus we must fix this using class weighting/SMOTE for oversampling

In [ ]:
#@title Examining categorical cardinality

#segregating categorical , binary and continuous value columns:
cat_cols=[c for c in df_train.columns if c.endswith('_cat')]
cont_cols = [c for c in df_train.columns
             if not c.endswith(('_cat', '_bin'))
             and not c.endswith('_missing')
             and c not in ['id', 'target']]
bin_cols=[c for c in df_train if c.endswith('_bin')]

print(df_train[cat_cols].nunique().sort_values(ascending=False))

ps_car_11_cat    104
ps_car_06_cat     18
ps_car_01_cat     12
ps_car_04_cat     10
ps_ind_05_cat      7
ps_car_09_cat      5
ps_ind_02_cat      4
ps_car_10_cat      3
ps_car_02_cat      3
ps_car_03_cat      3
ps_car_05_cat      3
ps_ind_04_cat      2
ps_car_07_cat      2
ps_car_08_cat      2
dtype: int64


We can use One-Hot Encoding for all the columns except ps_car_11_cat as encoding it would introduce 104 new features , and we would enter into the curse of dimensionality.

In [ ]:
#@title Encoding all the cols except ps_car_11_cat:
cats=[c for c in cat_cols if c!='ps_car_11_cat']
df_train = pd.get_dummies(df_train, columns=cats, drop_first=True)


In [ ]:
#@title Examining binary columns:

print(df_train[bin_cols].mean().sort_values())

ps_ind_10_bin     0.000370
ps_ind_13_bin     0.000921
ps_ind_11_bin     0.001637
ps_ind_12_bin     0.009243
ps_ind_17_bin     0.120845
ps_calc_15_bin    0.122450
ps_calc_20_bin    0.153246
ps_ind_18_bin     0.153614
ps_ind_08_bin     0.164157
ps_ind_09_bin     0.185604
ps_ind_07_bin     0.256930
ps_calc_18_bin    0.287161
ps_calc_19_bin    0.349102
ps_ind_06_bin     0.393308
ps_calc_17_bin    0.554258
ps_calc_16_bin    0.627844
ps_ind_16_bin     0.660912
dtype: float64


We see some cols have a mean ~0  , indicating that almost all the values in that column are zeroes , barely any ones. We can check for correlation w/target to decide if we drop the col or not

In [ ]:
corr=df_train[bin_cols].corrwith(df_train['target'])
print(corr)

ps_ind_06_bin    -0.033860
ps_ind_07_bin     0.033828
ps_ind_08_bin     0.013113
ps_ind_09_bin    -0.007969
ps_ind_10_bin     0.001024
ps_ind_11_bin     0.002106
ps_ind_12_bin     0.007845
ps_ind_13_bin     0.002259
ps_ind_16_bin    -0.027360
ps_ind_17_bin     0.036061
ps_ind_18_bin     0.004925
ps_calc_15_bin   -0.000626
ps_calc_16_bin    0.000647
ps_calc_17_bin   -0.000069
ps_calc_18_bin    0.000368
ps_calc_19_bin   -0.001592
ps_calc_20_bin   -0.001153
dtype: float64


We see that all the binary features have extremely weak correlation with the target. We can drop a lot of these cols as well.

In [ ]:
drop_bin_cols = [
    'ps_calc_15_bin', 'ps_calc_16_bin', 'ps_calc_17_bin',
    'ps_calc_18_bin', 'ps_calc_19_bin', 'ps_calc_20_bin',
    'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin'
]


df_train.drop(columns=drop_bin_cols, inplace=True)

In [ ]:
#@title Examining the other (continuous) features:

calc_cont_cols = [c for c in cont_cols if c.startswith('ps_calc')]
print(df_train[calc_cont_cols].corrwith(df_train['target']).sort_values())

ps_calc_12   -0.001321
ps_calc_08   -0.001066
ps_calc_13   -0.000450
ps_calc_07   -0.000177
ps_calc_06   -0.000003
ps_calc_04    0.000180
ps_calc_09    0.000355
ps_calc_05    0.000384
ps_calc_11    0.000434
ps_calc_10    0.000443
ps_calc_14    0.001215
ps_calc_01    0.001503
ps_calc_02    0.001562
ps_calc_03    0.001773
dtype: float64


Here we check if the continuous ps_calc_* features actually relate to target. If these come back near-zero too (like the binary ones did), we can drop the whole ps_calc group.

These are also all near zero values, thus we can drop them

In [ ]:
#@title Dropping ps_calc_* continuous cols
df_train.drop(columns=calc_cont_cols, inplace=True)

Now checking the remaining continuous/ordinal features (ind, reg, car groups) against target. Binning into deciles lets us see if there's a monotonic trend - a feature that consistently goes up or down across deciles is more likely to be useful for the model than one that stays flat.

In [ ]:
#@title Decile binning - continuous features vs target:

other_cont_cols = [c for c in cont_cols if not c.startswith('ps_calc')]

for col in other_cont_cols:
    df_train[f'{col}_decile'] = pd.qcut(df_train[col], 10, duplicates='drop')
    print(col)
    print(df_train.groupby(f'{col}_decile', observed=True)['target'].mean())
    print()

ps_ind_01
ps_ind_01_decile
(-0.001, 1.0]    0.032828
(1.0, 2.0]       0.036485
(2.0, 3.0]       0.040603
(3.0, 4.0]       0.044680
(4.0, 5.0]       0.039942
(5.0, 7.0]       0.041992
Name: target, dtype: float64

ps_ind_03
ps_ind_03_decile
(-0.001, 1.0]    0.038800
(1.0, 2.0]       0.030308
(2.0, 3.0]       0.029526
(3.0, 4.0]       0.033142
(4.0, 5.0]       0.038796
(5.0, 6.0]       0.041755
(6.0, 7.0]       0.041463
(7.0, 8.0]       0.038907
(8.0, 11.0]      0.038053
Name: target, dtype: float64

ps_ind_14
ps_ind_14_decile
(-0.001, 4.0]    0.035961
Name: target, dtype: float64

ps_ind_15
ps_ind_15_decile
(-0.001, 2.0]    0.043725
(2.0, 4.0]       0.038559
(4.0, 5.0]       0.038581
(5.0, 7.0]       0.038802
(7.0, 8.0]       0.035220
(8.0, 10.0]      0.032486
(10.0, 11.0]     0.030253
(11.0, 12.0]     0.030461
(12.0, 13.0]     0.030602
Name: target, dtype: float64

ps_reg_01
ps_reg_01_decile
(-0.001, 0.1]    0.028268
(0.1, 0.3]       0.029910
(0.3, 0.4]       0.034239
(0.4, 0.6]       

Thus ,we see that:
- cols like ps_car_13 , ps_reg_02 , ps_car_12 show the highest monotonic trend
and thus are strong predictive features.

- Cols like ps_ind_01 and ps_ind_03 have no clear pattern
- ps_ind_14 only produces one bin --> We can drop this column



Dropping the decile columns now that we've looked at them - they were just for this check, not meant to be kept as features.

In [ ]:
#@title Cleaning up the temporary decile columns:

decile_cols = [c for c in df_train.columns if c.endswith('_decile')]
df_train.drop(columns=decile_cols, inplace=True)

In [ ]:
df_train.drop(columns=['ps_ind_14'])

,id,target,ps_ind_01,ps_ind_03,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_car_14_missing,ps_reg_03_missing,ps_car_07_cat_missing,ps_ind_02_cat_2,ps_ind_02_cat_3,ps_ind_02_cat_4,ps_ind_04_cat_1,ps_ind_05_cat_1,ps_ind_05_cat_2,ps_ind_05_cat_3,ps_ind_05_cat_4,ps_ind_05_cat_5,ps_ind_05_cat_6,ps_car_01_cat_1,ps_car_01_cat_2,ps_car_01_cat_3,ps_car_01_cat_4,ps_car_01_cat_5,ps_car_01_cat_6,ps_car_01_cat_7,ps_car_01_cat_8,ps_car_01_cat_9,ps_car_01_cat_10,ps_car_01_cat_11,ps_car_02_cat_0,ps_car_02_cat_1,ps_car_03_cat_0,ps_car_03_cat_1,ps_car_04_cat_1,ps_car_04_cat_2,ps_car_04_cat_3,ps_car_04_cat_4,ps_car_04_cat_5,ps_car_04_cat_6,ps_car_04_cat_7,ps_car_04_cat_8,ps_car_04_cat_9,ps_car_05_cat_0,ps_car_05_cat_1,ps_car_06_cat_1,ps_car_06_cat_2,ps_car_06_cat_3,ps_car_06_cat_4,ps_car_06_cat_5,ps_car_06_cat_6,ps_car_06_cat_7,ps_car_06_cat_8,ps_car_06_cat_9,ps_car_06_cat_10,ps_car_06_cat_11,ps_car_06_cat_12,ps_car_06_cat_13,ps_car_06_cat_14,ps_car_06_cat_15,ps_car_06_cat_16,ps_car_06_cat_17,ps_car_07_cat_1,ps_car_08_cat_1,ps_car_09_cat_1,ps_car_09_cat_2,ps_car_09_cat_3,ps_car_09_cat_4,ps_car_10_cat_1,ps_car_10_cat_2
0,7,0,2,5,0,1,0,0,11,0,1,0,0.7,0.2,0.718070,12,2,0.400000,0.883679,0.370810,3.605551,0,0,0,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
1,9,0,1,7,0,0,1,0,3,0,0,1,0.8,0.4,0.766078,19,3,0.316228,0.618817,0.388716,2.449490,0,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,True,False,True,False,False,True,False
2,13,0,5,9,0,0,1,0,12,1,0,0,0.0,0.0,0.801561,60,1,0.316228,0.641586,0.347275,3.316625,0,1,0,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False,True,False,False,True,False
3,16,0,0,2,1,0,0,0,8,1,0,0,0.9,0.2,0.580948,104,1,0.374166,0.542949,0.294958,2.000000,0,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,True,False,False,True,False,True,False
4,17,0,0,0,1,0,0,0,9,1,0,0,0.7,0.6,0.840759,82,3,0.316070,0.565832,0.365103,2.000000,0,0,0,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595207,1488013,0,3,10,0,0,0,1,13,1,0,0,0.5,0.3,0.692820,31,3,0.374166,0.684631,0.385487,2.645751,0,0,0,False,False,False,False,False,Fa